## Importing Required Libraries

In this section, we import all the necessary libraries required for:
- File handling and data processing
- Natural Language Processing (NLP)
- Deep Learning using PyTorch

We also download required NLTK tokenizers for text preprocessing.

In [1]:
import os
import re
import zipfile
import urllib.request
from collections import Counter
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import nltk
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass
import sys
sys.path.append('..')

from datasets import load_dataset
import torch
from torch.utils.data import DataLoader

## Defining Constants and File Paths

We define important constants such as:
- GloVe embedding paths
- Embedding dimensions
- Special tokens
- Maximum sequence lengths for questions and answers

These values control how text data is processed and represented.

In [3]:
glove_dir = 'glove_data'
glove_file = os.path.join(GLOVE_DIR, 'glove.6B.100d.txt')
glove_zip  = os.path.join(GLOVE_DIR, 'glove.6B.zip')
glove_url  = 'https://nlp.stanford.edu/data/glove.6B.zip'
emb_dim  = 100

pad_token = '<PAD>'
unk_token = '<UNK>'

q_len_max = 40
a_len_max = 100

## Downloading GloVe Embeddings

GloVe (Global Vectors for Word Representation) provides pretrained word embeddings.

If not already present, we:
1. Download the embeddings
2. Extract them for use in our model

In [4]:
def download_glove():
    if not os.path.exists(glove_file):
        os.makedirs(glove_dir, exist_ok=True)
        print("Downloading GloVe...")
        urllib.request.urlretrieve(glove_url, glove_zip)
        with zipfile.ZipFile(glove_zip, 'r') as zip_ref:
            zip_ref.extractall(glove_dir)
        print("Download complete.")

## Loading GloVe Embeddings

We load pretrained word vectors into a dictionary.

Each word is mapped to a fixed-size numerical vector representation.

In [5]:
def load_glove(path=glove_file, embed_dim=emb_dim):
    glove = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove[word] = vector
    return glove

## Tokenizing Text

Tokenization converts raw text into smaller units (tokens).

We:
- Convert text to lowercase
- Extract words using regular expressions

In [6]:
def tokenize(text):
    text = text.lower()
    return re.findall(r'\b\w+\b', text)

## Building Vocabulary

We construct a vocabulary from training datasets.

Steps:
- Count word frequencies
- Assign unique indices to each word
- Add special tokens (<PAD>, <UNK>)

In [7]:
def build_vocab(wiki_train, trec_train):
    counter = Counter()

    for dataset in [wiki_train, trec_train]:
        for sample in dataset:
            counter.update(tokenize(sample['question']))
            counter.update(tokenize(sample['answer']))

    vocab = [pad_token,unk_token] + list(counter.keys())
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}

    return word2idx, idx2word

## Creating Embedding Matrix

We convert vocabulary words into vectors using GloVe.

If a word is not found in GloVe:
- It is initialized randomly

In [8]:
def build_embedding_matrix(word2idx, glove_vectors, emb_dim=emb_dim):
    matrix = np.random.normal(size=(len(word2idx), emb_dim))

    for word, idx in word2idx.items():
        if word in glove_vectors:
            matrix[idx] = glove_vectors[word]

    return torch.tensor(matrix, dtype=torch.float32)

## Encoding and Padding

We convert tokens into numerical indices and ensure fixed-length sequences by padding.

This is necessary for batch processing in neural networks.

In [9]:
def encode_and_pad(tokens, max_len, word2idx):
    encoded = [word2idx.get(t, word2idx[unk_token]) for t in tokens]
    if len(encoded) < max_len:
        encoded += [word2idx[pad_token]] * (max_len - len(encoded))
    return encoded[:max_len]

## Finding Question Tokens in Answers

We identify positions of question words inside the answer text.

This helps in learning alignment between question and answer.

In [10]:
def find_question_positions(q_tokens, a_tokens):
    positions = []
    for i, word in enumerate(a_tokens):
        if word in q_tokens:
            positions.append(i)
    return positions

## Creating Custom Dataset

We define a PyTorch Dataset class to:
- Process each sample
- Convert text into numerical format
- Return structured data for training

In [11]:
class QADataset(Dataset):
    def __init__(self, dataset, word2idx):
        self.dataset = dataset
        self.word2idx = word2idx

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        q_tokens = tokenize(sample['question'])
        a_tokens = tokenize(sample['answer'])

        q_encoded = encode_and_pad(q_tokens, q_len_max, self.word2idx)
        a_encoded = encode_and_pad(a_tokens, a_len_max, self.word2idx)

        return {
            'question': torch.tensor(q_encoded),
            'answer': torch.tensor(a_encoded),
            'q_len': len(q_tokens),
            'a_len': len(a_tokens),
            'label': torch.tensor(sample.get('label', 0), dtype=torch.float32)
        }

## Collate Function

The collate function is used to combine multiple samples into a batch.

It stacks tensors and prepares them for training.

In [17]:
def collate_fn(batch):
    return {
        'question': torch.stack([b['question'] for b in batch]),
        'answer': torch.stack([b['answer'] for b in batch]),
        'q_len': torch.tensor([b['q_len'] for b in batch]),
        'a_len': torch.tensor([b['a_len'] for b in batch]),
        'label': torch.stack([b['label'] for b in batch])
    }

## Environment Setup

We configure the environment and select the computation device:
- GPU (if available)
- Otherwise CPU

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## Loading Datasets

We load:
- WikiQA dataset
- TREC-QA dataset

These datasets are used for training and evaluation.

In [14]:
wiki_qa = load_dataset('wiki_qa')
trec_qa_raw = load_dataset('lucadiliello/trecqa')

print('WikiQA splits:', list(wiki_qa.keys()))
print('TREC-QA splits:', list(trec_qa_raw.keys()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/594k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/264k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.00M [00:00<?, ?B/s]

...

Generating dev split:   0%|          | 0/1364 [00:00<?, ? examples/s]

Generating dev_clean split:   0%|          | 0/1343 [00:00<?, ? examples/s]

Generating test_clean split:   0%|          | 0/1442 [00:00<?, ? examples/s]

Generating train_all split:   0%|          | 0/53417 [00:00<?, ? examples/s]

WikiQA splits: ['test', 'validation', 'train']
TREC-QA splits: ['train', 'test', 'dev', 'dev_clean', 'test_clean', 'train_all']


## Preparing Vocabulary and Embeddings

We:
1. Build vocabulary from datasets
2. Download GloVe embeddings
3. Create embedding matrix

In [15]:
word2idx, idx2word = build_vocab(wiki_qa['train'], trec_qa_raw['train'])
print(f'Vocabulary size: {len(word2idx)}')

download_glove()
glove_vectors = load_glove()
embedding_matrix = build_embedding_matrix(word2idx, glove_vectors)

Vocabulary size: 35657
Download complete.


## Creating DataLoaders

We create DataLoader objects for batching and efficient training.

We also inspect a sample datapoint to verify preprocessing.

In [18]:
BATCH_SIZE = 64

wiki_train_ds = QADataset(wiki_qa['train'], word2idx)

wiki_train_loader = DataLoader(
    wiki_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

print(f'WikiQA Train size: {len(wiki_train_ds)}')

sample = wiki_train_ds[0]
print('\nSample Keys:', sample.keys())
print('Question Length:', sample['q_len'])
print('Answer Length:', sample['a_len'])
print('Label:', sample['label'].item())

WikiQA Train size: 20360

Sample Keys: dict_keys(['question', 'answer', 'q_len', 'a_len', 'label'])
Question Length: 5
Answer Length: 9
Label: 0.0


In [19]:
trec_train_ds = QADataset(trec_qa_raw['train'], word2idx)

trec_train_loader = DataLoader(
    trec_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

print(f'TREC-QA Train size: {len(trec_train_ds)}')

trec_sample = trec_train_ds[0]
print('\nTREC Sample Keys:', trec_sample.keys())
print('Question Length:', trec_sample['q_len'])
print('Answer Length:', trec_sample['a_len'])
print('Label:', trec_sample['label'].item())

TREC-QA Train size: 5919

TREC Sample Keys: dict_keys(['question', 'answer', 'q_len', 'a_len', 'label'])
Question Length: 15
Answer Length: 16
Label: 1.0
